In [2]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict
from langchain_groq import ChatGroq
from dotenv import load_dotenv
from langgraph.checkpoint.memory import InMemorySaver

In [4]:
load_dotenv()

llm = ChatGroq(model="openai/gpt-oss-120b")

In [5]:
class JokeState(TypedDict):

    topic: str
    joke: str
    explanation: str

In [6]:
def generate_joke(state: JokeState):

    prompt = f'generate a joke on the topic {state["topic"]}'
    response = llm.invoke(prompt).content

    return {'joke': response}

In [7]:
def generate_explanation(state: JokeState):

    prompt = f'write an explanation for the joke - {state["joke"]}'
    response = llm.invoke(prompt).content

    return {'explanation': response}

In [8]:
graph = StateGraph(JokeState)

graph.add_node('generate_joke', generate_joke)
graph.add_node('generate_explanation', generate_explanation)

graph.add_edge(START, 'generate_joke')
graph.add_edge('generate_joke', 'generate_explanation')
graph.add_edge('generate_explanation', END)

checkpointer = InMemorySaver()  # It will store in RAM

workflow = graph.compile(checkpointer=checkpointer)  # Here we just give checkpointer so workflow can understand we have to save all states

In [9]:
config1 = {"configurable": {"thread_id": "1"}}
workflow.invoke({'topic':'pizza'}, config=config1) # When we use persisteance we have to give thread id cause it store workflow against thread id , here we tell that for this particuler workflow thread_id is 1

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the company was looking for someone with *dough* experience and a *slice* of talent! 🍕😄',
 'explanation': '### Why the Pizza Joke Works  \n\n**The joke:**  \n> *Why did the pizza apply for a job?*  \n> *Because it heard the company was looking for someone with **dough** experience and a **slice** of talent!* 🍕😄  \n\n#### 1. The Setup (the “Why did the pizza…?” line)  \n- The question sets up an absurd scenario: a pizza—a food item—trying to get a job.  \n- The absurdity primes the listener to expect a punchline that will involve a play on words related to pizza.\n\n#### 2. The Wordplay (the punchline)  \n\n| Word in the punchline | Normal meaning (in a job‑search context) | Pizza‑related meaning (the pun) |\n|-----------------------|--------------------------------------------|---------------------------------|\n| **dough**             | Slang for money; also “experience” (e.g., “having dough” can imp

In [10]:
workflow.get_state(config1) # Using thread id (config1) we can fetch finel state value

StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the company was looking for someone with *dough* experience and a *slice* of talent! 🍕😄', 'explanation': '### Why the Pizza Joke Works  \n\n**The joke:**  \n> *Why did the pizza apply for a job?*  \n> *Because it heard the company was looking for someone with **dough** experience and a **slice** of talent!* 🍕😄  \n\n#### 1. The Setup (the “Why did the pizza…?” line)  \n- The question sets up an absurd scenario: a pizza—a food item—trying to get a job.  \n- The absurdity primes the listener to expect a punchline that will involve a play on words related to pizza.\n\n#### 2. The Wordplay (the punchline)  \n\n| Word in the punchline | Normal meaning (in a job‑search context) | Pizza‑related meaning (the pun) |\n|-----------------------|--------------------------------------------|---------------------------------|\n| **dough**             | Slang for money; also “experience” (e.g., “ha

In [ ]:
list(workflow.get_state_history(config1)) # We can also get intermideate values and here also have to give thread_id

# It give before start value (empty {}) also give next value (start)
# before gen_joke value = pizza next value generate joke
# like wise..

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the company was looking for someone with *dough* experience and a *slice* of talent! 🍕😄', 'explanation': '### Why the Pizza Joke Works  \n\n**The joke:**  \n> *Why did the pizza apply for a job?*  \n> *Because it heard the company was looking for someone with **dough** experience and a **slice** of talent!* 🍕😄  \n\n#### 1. The Setup (the “Why did the pizza…?” line)  \n- The question sets up an absurd scenario: a pizza—a food item—trying to get a job.  \n- The absurdity primes the listener to expect a punchline that will involve a play on words related to pizza.\n\n#### 2. The Wordplay (the punchline)  \n\n| Word in the punchline | Normal meaning (in a job‑search context) | Pizza‑related meaning (the pun) |\n|-----------------------|--------------------------------------------|---------------------------------|\n| **dough**             | Slang for money; also “experience” (e.g., “h

In [12]:
config2 = {"configurable": {"thread_id": "2"}}
workflow.invoke({'topic':'pasta'}, config=config2)

{'topic': 'pasta',
 'joke': 'Why did the spaghetti get a promotion?  \n\nBecause it knew how to *pasta*blely handle any *sauce*-ial situation! 🍝',
 'explanation': '**What makes the joke funny?**  \n\n1. **The setup creates an expectation** – “Why did the spaghetti get a promotion?” sounds like a typical “knock‑knock” or “why‑did‑the‑chicken” joke where the punch‑line will explain a surprising, often literal, reason for the promotion.  \n\n2. **The punch‑line is built on two puns that hinge on the language of food and work:**  \n\n   - **“*pasta*blely”**  \n     * The word *pasta* (the family of noodles that includes spaghetti) is inserted into the adverb *possibly* → *pasta‑bly*.  \n     * The result sounds just like “possibly,” a word you’d use when talking about someone’s ability to handle tasks.  \n     * The humor comes from swapping a regular word for a food‑related one, making the sentence sound goofy while still being understandable.  \n\n   - **“*sauce*-ial”**  \n     * Here th

In [13]:
workflow.get_state(config2)

StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti get a promotion?  \n\nBecause it knew how to *pasta*blely handle any *sauce*-ial situation! 🍝', 'explanation': '**What makes the joke funny?**  \n\n1. **The setup creates an expectation** – “Why did the spaghetti get a promotion?” sounds like a typical “knock‑knock” or “why‑did‑the‑chicken” joke where the punch‑line will explain a surprising, often literal, reason for the promotion.  \n\n2. **The punch‑line is built on two puns that hinge on the language of food and work:**  \n\n   - **“*pasta*blely”**  \n     * The word *pasta* (the family of noodles that includes spaghetti) is inserted into the adverb *possibly* → *pasta‑bly*.  \n     * The result sounds just like “possibly,” a word you’d use when talking about someone’s ability to handle tasks.  \n     * The humor comes from swapping a regular word for a food‑related one, making the sentence sound goofy while still being understandable.  \n\n   - **“*sauce*-ial”*

In [15]:
list(workflow.get_state_history(config2))

[StateSnapshot(values={'topic': 'pasta', 'joke': 'Why did the spaghetti get a promotion?  \n\nBecause it knew how to *pasta*blely handle any *sauce*-ial situation! 🍝', 'explanation': '**What makes the joke funny?**  \n\n1. **The setup creates an expectation** – “Why did the spaghetti get a promotion?” sounds like a typical “knock‑knock” or “why‑did‑the‑chicken” joke where the punch‑line will explain a surprising, often literal, reason for the promotion.  \n\n2. **The punch‑line is built on two puns that hinge on the language of food and work:**  \n\n   - **“*pasta*blely”**  \n     * The word *pasta* (the family of noodles that includes spaghetti) is inserted into the adverb *possibly* → *pasta‑bly*.  \n     * The result sounds just like “possibly,” a word you’d use when talking about someone’s ability to handle tasks.  \n     * The humor comes from swapping a regular word for a food‑related one, making the sentence sound goofy while still being understandable.  \n\n   - **“*sauce*-ial”

## Time Travel

In [19]:
workflow.get_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1d4467-9ef3-68d5-8000-6a8285a37916"}})

StateSnapshot(values={'topic': 'pizza'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_id': '1f1d4467-9ef3-68d5-8000-6a8285a37916'}}, metadata={'source': 'loop', 'step': 0, 'parents': {}}, created_at='2026-10-30T09:44:18.371586+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d4467-9eed-64c0-bfff-bbce972629c1'}}, tasks=(PregelTask(id='33ecd4b3-7af0-3e98-8a94-035cc6613a5e', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result={'joke': 'Why did the pizza apply for a job?  \n\nBecause it heard the company was looking for someone with *dough* experience and a *slice* of talent! 🍕😄'}),), interrupts=())

In [20]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1d4467-9ef3-68d5-8000-6a8285a37916"}}) # We use None to start with where we stop(resume)

{'topic': 'pizza',
 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to earn some *dough* while working on its *crust*‑ometrics! 🍕😄',
 'explanation': '**What makes the joke funny?**  \n\n1. **Double‑meaning of “dough”**  \n   * *Literal*: Pizza is made from a ball of **dough** that’s stretched into a crust.  \n   * *Slang*: “Dough” is a common colloquial term for **money** (“I need more dough”).  \n   The punch‑line says the pizza wants to “earn some *dough*,” so we hear both the idea of a pizza earning cash and a pizza literally making more of its own dough.\n\n2. **Play on “crust‑ometrics”**  \n   * *Crust*: The browned edge of a pizza slice.  \n   * *‑metrics*: A suffix meaning “the science or measurement of something” (as in *metrics*, *econometrics*, *psychometrics*, etc.).  \n   By coining the nonsense word **crust‑ometrics**, the joke pretends the pizza is studying or improving its own crust, just as a person might work on a professional skill set or a statistica

In [21]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to earn some *dough* while working on its *crust*‑ometrics! 🍕😄', 'explanation': '**What makes the joke funny?**  \n\n1. **Double‑meaning of “dough”**  \n   * *Literal*: Pizza is made from a ball of **dough** that’s stretched into a crust.  \n   * *Slang*: “Dough” is a common colloquial term for **money** (“I need more dough”).  \n   The punch‑line says the pizza wants to “earn some *dough*,” so we hear both the idea of a pizza earning cash and a pizza literally making more of its own dough.\n\n2. **Play on “crust‑ometrics”**  \n   * *Crust*: The browned edge of a pizza slice.  \n   * *‑metrics*: A suffix meaning “the science or measurement of something” (as in *metrics*, *econometrics*, *psychometrics*, etc.).  \n   By coining the nonsense word **crust‑ometrics**, the joke pretends the pizza is studying or improving its own crust, just as a person might work on a professional skill

## Updating State

In [22]:
workflow.update_state({"configurable": {"thread_id": "1", "checkpoint_id": "1f1d4467-9ef3-68d5-8000-6a8285a37916", "checkpoint_ns": ""}}, {'topic':'samosa'})

{'configurable': {'thread_id': '1',
  'checkpoint_ns': '',
  'checkpoint_id': '1f1d44d0-5f66-6574-8001-f53dac282fa4'}}

In [23]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa'}, next=('generate_joke',), config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d44d0-5f66-6574-8001-f53dac282fa4'}}, metadata={'source': 'update', 'step': 1, 'parents': {}}, created_at='2026-10-30T10:31:10.280003+00:00', parent_config={'configurable': {'thread_id': '1', 'checkpoint_ns': '', 'checkpoint_id': '1f1d4467-9ef3-68d5-8000-6a8285a37916'}}, tasks=(PregelTask(id='61a0212d-5042-e6ed-1131-1612c0f58a73', name='generate_joke', path=('__pregel_pull', 'generate_joke'), error=None, interrupts=(), state=None, result=None),), interrupts=()),
 StateSnapshot(values={'topic': 'pizza', 'joke': 'Why did the pizza apply for a job?\n\nBecause it wanted to earn some *dough* while working on its *crust*‑ometrics! 🍕😄', 'explanation': '**What makes the joke funny?**  \n\n1. **Double‑meaning of “dough”**  \n   * *Literal*: Pizza is made from a ball of **dough** that’s stretched into a crust.  \n   * *Slang*: “Dough” is a commo

In [24]:
workflow.invoke(None, {"configurable": {"thread_id": "1", "checkpoint_id": "1f1d44d0-5f66-6574-8001-f53dac282fa4"}})

{'topic': 'samosa',
 'joke': 'Why did the samosa go to therapy?\n\nBecause it was tired of being *filled* with emotions and wanted to *crumble* under pressure! 😄',
 'explanation': '**Breaking Down the Joke**\n\n| Part of the joke | What it literally means | The wordplay / double meaning |\n|------------------|------------------------|--------------------------------|\n| **“Why did the samosa go to therapy?”** | Sets up a classic “Why did X do Y?” format, inviting a punch‑line that explains an odd behavior. | The absurd image of a fried pastry seeking professional help creates the comedic premise. |\n| **“Because it was tired of being *filled* with emotions…”** | A samosa is literally *filled* with a mixture of potatoes, peas, spices, etc. | The word **filled** is used both for the physical stuffing of the snack and for the idiom “filled with emotions,” meaning overwhelmed by feelings. |\n| **“…and wanted to *crumble* under pressure!”** | When a samosa is over‑cooked or mishandled it ca

In [25]:
list(workflow.get_state_history(config1))

[StateSnapshot(values={'topic': 'samosa', 'joke': 'Why did the samosa go to therapy?\n\nBecause it was tired of being *filled* with emotions and wanted to *crumble* under pressure! 😄', 'explanation': '**Breaking Down the Joke**\n\n| Part of the joke | What it literally means | The wordplay / double meaning |\n|------------------|------------------------|--------------------------------|\n| **“Why did the samosa go to therapy?”** | Sets up a classic “Why did X do Y?” format, inviting a punch‑line that explains an odd behavior. | The absurd image of a fried pastry seeking professional help creates the comedic premise. |\n| **“Because it was tired of being *filled* with emotions…”** | A samosa is literally *filled* with a mixture of potatoes, peas, spices, etc. | The word **filled** is used both for the physical stuffing of the snack and for the idiom “filled with emotions,” meaning overwhelmed by feelings. |\n| **“…and wanted to *crumble* under pressure!”** | When a samosa is over‑cooked

## Fault Tolerance

In [7]:
from langgraph.graph import StateGraph, END
from langgraph.checkpoint.memory import InMemorySaver
from typing import TypedDict
import time

In [8]:
# 1. Define the state
class CrashState(TypedDict):
    input: str
    step1: str
    step2: str

In [9]:
# 2. Define steps
def step_1(state: CrashState) -> CrashState:
    print("✅ Step 1 executed")
    return {"step1": "done", "input": state["input"]}

def step_2(state: CrashState) -> CrashState:
    print("⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)")
    time.sleep(1000)  # Simulate long-running hang
    return {"step2": "done"}

def step_3(state: CrashState) -> CrashState:
    print("✅ Step 3 executed")
    return {"done": True}

In [10]:
# 3. Build the graph
builder = StateGraph(CrashState)
builder.add_node("step_1", step_1)
builder.add_node("step_2", step_2)
builder.add_node("step_3", step_3)

builder.set_entry_point("step_1")
builder.add_edge("step_1", "step_2")
builder.add_edge("step_2", "step_3")
builder.add_edge("step_3", END)

checkpointer = InMemorySaver()
graph = builder.compile(checkpointer=checkpointer)

In [ ]:
try:
    print("▶️ Running graph: Please manually interrupt during Step 2...")
    graph.invoke({"input": "start"}, config={"configurable": {"thread_id": 'thread-1'}})
except KeyboardInterrupt:
    print("❌ Kernel manually interrupted (crash simulated).")

▶️ Running graph: Please manually interrupt during Step 2...
✅ Step 1 executed
⏳ Step 2 hanging... now manually interrupt from the notebook toolbar (STOP button)


In [ ]:
# 6. Re-run to show fault-tolerant resume
print("\n🔁 Re-running the graph to demonstrate fault tolerance...")
final_state = graph.invoke(None, config={"configurable": {"thread_id": 'thread-1'}})
print("\n✅ Final State:", final_state)

In [ ]:
list(graph.get_state_history({"configurable": {"thread_id": 'thread-1'}}))